In [11]:
# import libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# === Load wildfire weather data ===
wildfire_weather_path = '../data/processed_data/Wildfire_Weather_2020_2024.csv'
wildfire_df = pd.read_csv(wildfire_weather_path)

# Ensure coordinates are numeric
wildfire_df['longitude'] = pd.to_numeric(wildfire_df['longitude'], errors='coerce')
wildfire_df['latitude'] = pd.to_numeric(wildfire_df['latitude'], errors='coerce')

# Drop rows without valid coordinates
wildfire_df = wildfire_df.dropna(subset=['longitude', 'latitude'])

# === Create GeoDataFrame from wildfire locations ===
geometry = [Point(xy) for xy in zip(wildfire_df['longitude'], wildfire_df['latitude'])]
wildfire_gdf = gpd.GeoDataFrame(wildfire_df, geometry=geometry, crs="EPSG:4326")

# === Load GACC shapefile ===
gacc_gdf = gpd.read_file('../data/raw_data/gacc_boundaries/National_GACC_Final_20250113.shp')
gacc_gdf = gacc_gdf.rename(columns={'GACCName': 'gacc'})

# Reproject wildfire points if needed
if wildfire_gdf.crs != gacc_gdf.crs:
    wildfire_gdf = wildfire_gdf.to_crs(gacc_gdf.crs)

# === Spatial join to find GACC regions (fixed) ===
wildfire_with_gacc = gpd.sjoin(
    wildfire_gdf,
    gacc_gdf[['geometry', 'gacc']],
    how='left',
    predicate='within',
    lsuffix='',
    rsuffix='_gacc'  # prevents index_right collision
)

# === Assign GACC for Hawaii manually if missing ===
hawaii_mask = (
    (wildfire_with_gacc['latitude'] >= 18) & (wildfire_with_gacc['latitude'] <= 23) &
    (wildfire_with_gacc['longitude'] >= -161) & (wildfire_with_gacc['longitude'] <= -154)
)
wildfire_with_gacc.loc[hawaii_mask, 'gacc'] = 'Hawaii Coordination Center'

# === Save updated CSV ===
output_path = '../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv'
wildfire_with_gacc.drop(columns='geometry').to_csv(output_path, index=False)

# === Summary ===
gacc_regions = wildfire_with_gacc['gacc'].dropna().unique()
print(f"🌎 Wildfire weather records tagged with {len(gacc_regions)} unique GACC regions.")
print("Regions:", gacc_regions)
print(f"✅ Saved: {output_path}")

🌎 Wildfire weather records tagged with 11 unique GACC regions.
Regions: ['Southern Area Coordination Center' 'Southwest Area Coordination Center'
 'Rocky Mountain Area Coordination Center'
 'Eastern Area Coordination Center'
 'Northern California Geographic Area Coordination Center'
 'Northwest Interagency Coordination Center'
 'Northern Rockies Coordination Center' 'Great Basin Coordination Center'
 'Southern California Coordination Center'
 'Alaska Interagency Coordination Center' 'Hawaii Coordination Center']
✅ Saved: ../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv


In [12]:
# Count the number of wildfire records per GACC region
gacc_counts = wildfire_with_gacc['gacc'].value_counts(dropna=True)

print("🔥 Wildfire counts by GACC region:")
print(gacc_counts)

🔥 Wildfire counts by GACC region:
gacc
Southern Area Coordination Center                          14634
Rocky Mountain Area Coordination Center                     4026
Northwest Interagency Coordination Center                   2833
Alaska Interagency Coordination Center                      2262
Northern Rockies Coordination Center                        1921
Eastern Area Coordination Center                            1695
Southwest Area Coordination Center                          1438
Northern California Geographic Area Coordination Center     1147
Great Basin Coordination Center                             1118
Southern California Coordination Center                      981
Hawaii Coordination Center                                    23
Name: count, dtype: int64
